# Low-rank LoRA prototype in PEFT convention

This notebook uses the same factor ordering as PEFT LoRA: `A` has shape `(r, d_in)`, `B` has shape `(d_out, r)`, and the low-rank update is `W = B @ A`. This matches PEFT's module order `lora_B(lora_A(x))`.


In [ ]:
from dataclasses import dataclass
from typing import List, Tuple

import numpy as np
import numpy.linalg as npl
import matplotlib.pyplot as plt


In [ ]:
# =============================
# Data generation (W = B A)
# =============================

def generate_lowrank_matrix(
    d_out: int,
    d_in: int,
    r: int,
    noise_std: float = 0.0,
    *,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Generate B0 (d_out x r), A0 (r x d_in), and W = B0 A0 + noise."""
    B0 = rng.standard_normal((d_out, r))
    A0 = rng.standard_normal((r, d_in))
    W = B0 @ A0
    if noise_std > 0:
        W = W + noise_std * rng.standard_normal((d_out, d_in))
    return B0, A0, W


def generate_with_spectrum(
    d_out: int,
    d_in: int,
    r: int,
    *,
    kappa: float = 1e6,
    kind: str = "geom",
    noise_std: float = 0.0,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Generate W = B0 A0 with prescribed singular-value decay."""
    Q_out, _ = np.linalg.qr(rng.standard_normal((d_out, r)))
    Q_in, _ = np.linalg.qr(rng.standard_normal((d_in, r)))

    if kind == "geom":
        if r == 1:
            s = np.array([1.0])
        else:
            s = kappa ** (-np.linspace(0.0, 1.0, r))
    elif kind == "power":
        p = 1.5
        s = 1.0 / (np.arange(1, r + 1) ** p)
        s /= s[0]
    else:
        raise ValueError("kind must be 'geom' or 'power'")

    B0 = Q_out * s[None, :]
    A0 = Q_in.T
    W = B0 @ A0
    if noise_std > 0:
        W = W + noise_std * rng.standard_normal((d_out, d_in))
    return B0, A0, W, s


# =============================
# Objective + gradients (W = B A)
# =============================

def objective(B: np.ndarray, A: np.ndarray, W: np.ndarray) -> float:
    E = B @ A - W
    return 0.5 * float(np.sum(E * E))


def gradients(B: np.ndarray, A: np.ndarray, W: np.ndarray) -> Tuple[np.ndarray, np.ndarray, float]:
    E = B @ A - W
    grad_B = E @ A.T
    grad_A = B.T @ E
    loss = 0.5 * float(np.sum(E * E))
    return grad_B, grad_A, loss


def rmse_full(B: np.ndarray, A: np.ndarray, W: np.ndarray) -> float:
    E = B @ A - W
    return float(np.sqrt(np.mean(E * E)))


# =============================
# Init + gradient methods
# =============================

def init_factors(
    d_out: int,
    d_in: int,
    r: int,
    *,
    rng: np.random.Generator,
    scale: float = 0.01,
) -> Tuple[np.ndarray, np.ndarray]:
    B = scale * rng.standard_normal((d_out, r))
    A = scale * rng.standard_normal((r, d_in))
    return B, A


def train_gd(
    W: np.ndarray,
    r: int,
    *,
    steps: int = 2000,
    lr: float = 1e-2,
    seed: int = 0,
    verbose_every: int = 100,
) -> Tuple[np.ndarray, np.ndarray, List[float]]:
    """Vanilla GD on 0.5 ||W - B A||_F^2."""
    d_out, d_in = W.shape
    rng = np.random.default_rng(seed)
    B, A = init_factors(d_out, d_in, r, rng=rng)

    if verbose_every > 0:
        print(f"Initial RMSE: {rmse_full(B, A, W):.6e}")

    history: List[float] = []
    for t in range(1, steps + 1):
        gB, gA, loss = gradients(B, A, W)
        B -= lr * gB
        A -= lr * gA
        history.append(loss)
        if verbose_every > 0 and (t % verbose_every == 0 or t == 1):
            print(f"step={t:5d} loss={loss:.6e}")
    return B, A, history


def train_gd_scaled(
    W: np.ndarray,
    r: int,
    *,
    steps: int = 2000,
    lr: float = 1.0,
    seed: int = 0,
    verbose_every: int = 100,
    ridge: float = 1e-12,
) -> Tuple[np.ndarray, np.ndarray, List[float]]:
    """Block-preconditioned GD in PEFT convention."""
    d_out, d_in = W.shape
    rng = np.random.default_rng(seed)
    B, A = init_factors(d_out, d_in, r, rng=rng)
    I = np.eye(r)

    if verbose_every > 0:
        print(f"Initial RMSE: {rmse_full(B, A, W):.6e}")

    history: List[float] = []
    for t in range(1, steps + 1):
        gB, gA, loss = gradients(B, A, W)

        S_B = B.T @ B
        S_A = A @ A.T
        P_B = npl.pinv(S_B + ridge * I)
        P_A = npl.pinv(S_A + ridge * I)

        # Same orientation as lora_playground.optim.ScaledLoRA:
        #   Delta A = -lr * S_B^{-1} grad_A
        #   Delta B = -lr * grad_B S_A^{-1}
        B -= lr * (gB @ P_A)
        A -= lr * (P_B @ gA)

        history.append(loss)
        if verbose_every > 0 and t % verbose_every == 0:
            print(f"step={t:5d} loss={loss:.6e}", flush=True)
    return B, A, history


def train_dual_gd_scaled(
    W: np.ndarray,
    r: int,
    *,
    steps: int = 2000,
    lr: float = 1.0,
    seed: int = 0,
    verbose_every: int = 100,
    ridge: float = 1e-12,
) -> Tuple[np.ndarray, np.ndarray, List[float]]:
    """Projected block-preconditioned GD for 0.5 ||B A - W||_F^2."""
    d_out, d_in = W.shape
    rng = np.random.default_rng(seed)
    B, A = init_factors(d_out, d_in, r, rng=rng)
    I = np.eye(r)

    if verbose_every > 0:
        print(f"Initial RMSE: {rmse_full(B, A, W):.6e}")

    history: List[float] = []
    for t in range(1, steps + 1):
        gB, gA, loss = gradients(B, A, W)

        S_B = B.T @ B
        S_A = A @ A.T
        P_B = npl.pinv(S_B + ridge * I)
        P_A = npl.pinv(S_A + ridge * I)

        B_projector = np.eye(d_out) - B @ P_B @ B.T
        A_projector = np.eye(d_in) - A.T @ P_A @ A

        B -= lr * (B_projector @ gB @ P_A)
        A -= lr * (P_B @ gA @ A_projector)

        history.append(loss)
        if verbose_every > 0 and t % verbose_every == 0:
            print(f"step={t:5d} loss={loss:.6e}", flush=True)
    return B, A, history


# =============================
# Gauss-Newton (PEFT convention)
# =============================

def _solve_right(M: np.ndarray, G: np.ndarray) -> np.ndarray:
    """Return X = M G^{-1} without forming G^{-1}."""
    return npl.solve(G.T, M.T).T


def _solve_left(G: np.ndarray, M: np.ndarray) -> np.ndarray:
    """Return X = G^{-1} M without forming G^{-1}."""
    return npl.solve(G, M)


def _sylvester_symmetric(
    alpha: float,
    S_B: np.ndarray,
    S_A: np.ndarray,
    C: np.ndarray,
    eps: float = 0.0,
) -> np.ndarray:
    """Solve alpha * S_B Y + Y S_A = -C for Y, with symmetric S_B and S_A."""
    lam_A, Q_A = npl.eigh(S_A)
    lam_B, Q_B = npl.eigh(S_B)

    if eps > 0:
        lam_A = lam_A + eps
        lam_B = lam_B + eps

    Cprime = Q_B.T @ C @ Q_A
    denom = lam_B[:, None] * alpha + lam_A[None, :]
    Yprime = -Cprime / denom
    return Q_B @ Yprime @ Q_A.T


def gauss_newton_step(
    B: np.ndarray,
    A: np.ndarray,
    W: np.ndarray,
    *,
    ridge: float = 1e-12,
    wB: float = 1.0,
    wA: float = 1.0,
) -> Tuple[np.ndarray, np.ndarray]:
    """One GN step (Delta B, Delta A) for 0.5 ||B A - W||_F^2."""
    E = B @ A - W
    S_A = A @ A.T
    S_B = B.T @ B
    C = B.T @ E @ A.T

    if ridge > 0:
        S_A = S_A + ridge * np.eye(S_A.shape[0])
        S_B = S_B + ridge * np.eye(S_B.shape[0])

    alpha = wA / wB
    Y = _sylvester_symmetric(alpha, S_B, S_A, C, eps=0.0)
    X = alpha * Y

    dB = -_solve_right(E @ A.T + B @ X, S_A)
    dA = -_solve_left(S_B, B.T @ E + Y @ A)
    return dB, dA


def train_gn(
    W: np.ndarray,
    r: int,
    *,
    steps: int = 200,
    seed: int = 0,
    verbose_every: int = 20,
    ridge: float = 1e-12,
) -> Tuple[np.ndarray, np.ndarray, List[float]]:
    """Unweighted Gauss-Newton. Complexity per step: O(d_out r^2 + d_in r^2 + r^3)."""
    d_out, d_in = W.shape
    rng = np.random.default_rng(seed)
    B, A = init_factors(d_out, d_in, r, rng=rng)

    if verbose_every > 0:
        print(f"Initial RMSE: {rmse_full(B, A, W):.6e}")

    hist: List[float] = []
    for t in range(1, steps + 1):
        dB, dA = gauss_newton_step(B, A, W, ridge=ridge, wB=1.0, wA=1.0)
        B += dB
        A += dA
        loss = objective(B, A, W)
        hist.append(loss)
        if verbose_every > 0 and (t % verbose_every == 0 or t == 1):
            print(f"[GN] step={t:4d} loss={loss:.6e}  RMSE={rmse_full(B, A, W):.6e}")
    return B, A, hist


def train_gn_weighted(
    W: np.ndarray,
    r: int,
    *,
    steps: int = 200,
    seed: int = 0,
    verbose_every: int = 20,
    ridge: float = 1e-12,
) -> Tuple[np.ndarray, np.ndarray, List[float]]:
    """Weighted-min-norm Gauss-Newton using PEFT factor dimensions."""
    d_out, d_in = W.shape
    rng = np.random.default_rng(seed)
    B, A = init_factors(d_out, d_in, r, rng=rng)

    if verbose_every > 0:
        print(f"Initial RMSE: {rmse_full(B, A, W):.6e}")

    # B is d_out x r and A is r x d_in. This mirrors the old LR notebook's
    # weighted metric after mapping L -> B and R -> A.
    wB, wA = 1.0 / r, 1.0 / d_in

    hist: List[float] = []
    for t in range(1, steps + 1):
        dB, dA = gauss_newton_step(B, A, W, ridge=ridge, wB=wB, wA=wA)
        B += dB
        A += dA
        loss = objective(B, A, W)
        hist.append(loss)
        if verbose_every > 0 and (t % verbose_every == 0 or t == 1):
            print(f"[GNw] step={t:4d} loss={loss:.6e}  RMSE={rmse_full(B, A, W):.6e}")
    return B, A, hist


# =============================
# Demo / quick test
# =============================

@dataclass
class Config:
    d_out: int = 80
    d_in: int = 60
    r_true: int = 5
    r_fit: int = 5
    noise_std: float = 0.05
    steps: int = 200
    lr: float = 1e-2
    seed: int = 42
    verbose_every: int = 20
    use_spectrum: bool = False
    spectrum_kind: str = "geom"
    kappa: float = 1e6
    method: str = "gn"
    ridge: float = 1e-12


In [ ]:
def run_experiment(cfg: Config):
    rng = np.random.default_rng(cfg.seed)
    if cfg.use_spectrum:
        B0, A0, W, s = generate_with_spectrum(
            cfg.d_out,
            cfg.d_in,
            cfg.r_true,
            kappa=cfg.kappa,
            kind=cfg.spectrum_kind,
            noise_std=cfg.noise_std,
            rng=rng,
        )
        print(f"constructed spectrum: sigma[0]={s[0]:.2e}, sigma[-1]={s[-1]:.2e}, cond={s[0] / s[-1]:.2e}")
    else:
        B0, A0, W = generate_lowrank_matrix(
            cfg.d_out,
            cfg.d_in,
            cfg.r_true,
            cfg.noise_std,
            rng=rng,
        )

    if cfg.method == "gd":
        B, A, hist = train_gd(
            W,
            r=cfg.r_fit,
            steps=cfg.steps,
            lr=cfg.lr,
            seed=cfg.seed + 1,
            verbose_every=cfg.verbose_every,
        )
    elif cfg.method == "scaled":
        B, A, hist = train_gd_scaled(
            W,
            r=cfg.r_fit,
            steps=cfg.steps,
            lr=cfg.lr,
            seed=cfg.seed + 1,
            verbose_every=cfg.verbose_every,
            ridge=cfg.ridge,
        )
    elif cfg.method == "dual":
        B, A, hist = train_dual_gd_scaled(
            W,
            r=cfg.r_fit,
            steps=cfg.steps,
            lr=cfg.lr,
            seed=cfg.seed + 1,
            verbose_every=cfg.verbose_every,
            ridge=cfg.ridge,
        )
    elif cfg.method == "gn":
        B, A, hist = train_gn(
            W,
            r=cfg.r_fit,
            steps=cfg.steps,
            seed=cfg.seed + 1,
            verbose_every=cfg.verbose_every,
            ridge=cfg.ridge,
        )
    elif cfg.method == "gn_weighted":
        B, A, hist = train_gn_weighted(
            W,
            r=cfg.r_fit,
            steps=cfg.steps,
            seed=cfg.seed + 1,
            verbose_every=cfg.verbose_every,
            ridge=cfg.ridge,
        )
    else:
        raise ValueError(f"Invalid method: {cfg.method}")

    print("--- Results ---")
    print(f"final loss: {hist[-1]:.6e}")
    print(f"RMSE: {rmse_full(B, A, W):.6e}")

    plt.plot(hist)
    plt.yscale("log")
    plt.xlabel("Iteration")
    plt.ylabel("Loss")
    plt.title(f"PEFT convention: {cfg.method}")
    plt.show()

    return B, A, hist


In [ ]:
sample_cfg = Config(lr=0.01, verbose_every=100, steps=2000, noise_std=0.0,
                    use_spectrum=True, method="scaled")
run_experiment(sample_cfg)


In [ ]:
sample_cfg = Config(lr=0.01, verbose_every=100, steps=2000, noise_std=0.0,
                    use_spectrum=True, method="gd")
run_experiment(sample_cfg)


In [ ]:
sample_cfg = Config(lr=0.01, verbose_every=10, steps=100, noise_std=0.0,
                    use_spectrum=True, method="gn")
run_experiment(sample_cfg)


In [ ]:
sample_cfg = Config(lr=0.01, verbose_every=1, steps=20, noise_std=0.0,
                    use_spectrum=True, method="gn_weighted")
run_experiment(sample_cfg)
